In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free
from constants import CASH, COMMISSION, SESSIONS

TABLE_NAME = "f1_session"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def session_mask(index, session):
    start = pd.Timestamp(session["start"]).time()
    end = pd.Timestamp(session["end"]).time()
    tod = np.array(index.to_series().dt.time.values)
    if start <= end:
        return (tod >= start) & (tod < end)
    return (tod >= start) | (tod < end)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "long": long_mask,
        "short": short_mask,
    })

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD, session=None):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal = signals["long"].to_numpy(dtype=bool)
    short_signal = signals["short"].to_numpy(dtype=bool)

    n = len(data)
    in_session = np.ones(n, dtype=bool)
    session_end = np.zeros(n, dtype=bool)
    if session is not None:
        in_session = session_mask(data.index, session)
        next_in = np.empty(n, dtype=bool)
        next_in[:-1] = in_session[1:]
        next_in[-1] = False
        session_end = in_session & ~next_in

    long_signal = long_signal & in_session & ~session_end
    short_signal = short_signal & in_session & ~session_end
    exits_long = short_signal | session_end
    exits_short = long_signal | session_end

    close_np = data["Close"].to_numpy(dtype=np.float64)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=exits_long,
        short_entries=short_signal,
        short_exits=exits_short,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, sl_mult, rr_ratio, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD, session=None):
    session_name = session["name"] if session else None
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period, session=session)
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [2]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M15.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(
    data,
    n1=5,
    rr_ratio=2,
    atr_period=18,
    session=SESSIONS[1],
    sl_mult=3,
    use_vwap=1,
    smooth_period=12,
)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M15__cache_v1.parquet
Start                                                 0
End                                              139917
Period                               1457 days 11:30:00
Start Value                                     50000.0
End Value                                  50007.464764
Total Return [%]                                0.01493
Benchmark Return [%]                         184.221285
Max Gross Exposure [%]                         0.203878
Total Fees Paid                               56.402929
Max Drawdown [%]                               0.030839
Max Drawdown Duration                 724 days 06:45:00
Total Trades                                       1880
Total Closed Trades                                1880
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                   41.43617
Best Trade [%]                                 5.487051

In [4]:
# portfolio.plot()
portfolio.value().vbt.plot()

FigureWidget({
    'data': [{'showlegend': False,
              'type': 'scatter',
              'uid': '122b6951-d8cb-490c-ab47-8e56bb7a2962',
              'x': {'bdata': ('AAAAAAEAAAACAAAAAwAAAAQAAAAFAA' ... 'CJIgIAiiICAIsiAgCMIgIAjSICAA=='),
                    'dtype': 'i4'},
              'y': {'bdata': ('AAAAAABq6EAAAAAAAGroQAAAAAAAau' ... 'jf7mroQP0BWN/uauhA/QFY3+5q6EA='),
                    'dtype': 'f8'}}],
    'layout': {'height': 350,
               'legend': {'orientation': 'h',
                          'traceorder': 'normal',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'margin': {'b': 30, 'l': 30, 'r': 30, 't': 30},
               'template': '...',
               'width': 700}
})